# WELL Generator Equidistribution

WELL (Well Equidistributed Long-period Linear) generators, implemented
as carry-type generators. We test WELL19937 and WELL44497 with and
without tempering, using the lattice method.

In [ ]:
import regpoly._regpoly_cpp as _cpp
from regpoly.generateur import Generateur
from regpoly.transformation import Transformation
from regpoly.combinaison import Combinaison
from regpoly.tests.equidistribution_test import (
    EquidistributionTest, METHOD_DUALLATTICE
)

INT_MAX = 2**31 - 1

## 1. WELL19937a (without tempering)

Carry generator with $w=32$, $r=624$, $p=31$, $k = 32 \times 624 - 31 = 19937$.

In [ ]:
well19937_params = {
    "w": 32, "r": 624, "p": 31,
    "m1": 70, "m2": 179, "m3": 449,
    "mat_types": [0, 0, 3, 0, 1, 0, 0, 0],
    "mat_pi":  [-25,0,0, 27,0,0, 9,0,0, 1,0,0, 0,0,0, -9,0,0, -21,0,0, 21,0,0],
    "mat_pu":  [0]*24,
}

gen_19937 = Generateur(_cpp.create_generator("carry", well19937_params, 32))
print(f"{gen_19937.name()}: k={gen_19937.k}, L={gen_19937.L}")
gen_19937.display()

In [ ]:
comb = Combinaison(J=1, Lmax=32)
comb.components[0].add_gen(gen_19937)
comb.reset()

test = EquidistributionTest(L=32, delta=[INT_MAX]*33,
                            mse=INT_MAX, method=METHOD_DUALLATTICE)
result = test.run(comb)

print(f"WELL19937a (no tempering): dimension gaps sum = {result.se}")
result.display_table(comb, 'l')

## 2. WELL19937c (with tempering)

Same generator with Matsumoto-Kurita type I tempering.

In [ ]:
temper = Transformation("tempMK", {
    "w": 32, "eta": 7, "mu": 15,
    "b": 0xe46e1700, "c": 0x9b868000
})

comb_t = Combinaison(J=1, Lmax=32)
comb_t.components[0].add_gen(gen_19937)
comb_t.components[0].add_trans(temper)
comb_t.reset()

result_t = test.run(comb_t)

print(f"WELL19937c (with tempering): dimension gaps sum = {result_t.se}")
result_t.display_table(comb_t, 'l')

## 3. WELL44497a (without tempering)

Carry generator with $w=32$, $r=1391$, $p=15$, $k = 32 \times 1391 - 15 = 44497$.

In [ ]:
well44497_params = {
    "w": 32, "r": 1391, "p": 15,
    "m1": 23, "m2": 481, "m3": 229,
    "mat_types": [0, 0, 0, 3, 1, 0, 5, 1],
    "mat_pi":  [-24,0,0, 30,0,0, -10,0,0, -26,0,0, 0,0,0, 20,0,0, 9,0,0, 0,0,0],
    "mat_pu":  [0,0,0, 0,0,0, 0,0,0, 0,0,0, 0,0,0, 0,0,0,
                0xb729fcec, 0xfbffffff, 0x00020000, 0,0,0],
}

gen_44497 = Generateur(_cpp.create_generator("carry", well44497_params, 32))
print(f"{gen_44497.name()}: k={gen_44497.k}, L={gen_44497.L}")
gen_44497.display()

In [ ]:
comb44 = Combinaison(J=1, Lmax=32)
comb44.components[0].add_gen(gen_44497)
comb44.reset()

result44 = test.run(comb44)

print(f"WELL44497a (no tempering): dimension gaps sum = {result44.se}")
result44.display_table(comb44, 'l')

## 4. Comparison

In [ ]:
print(f"{'Generator':<25}  {'k':>6}  {'Gaps Sum':>10}")
print("-" * 47)
print(f"{'WELL19937a (raw)':<25}  {19937:>6}  {result.se:>10}")
print(f"{'WELL19937c (tempered)':<25}  {19937:>6}  {result_t.se:>10}")
print(f"{'WELL44497a (raw)':<25}  {44497:>6}  {result44.se:>10}")